In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

sys.path.append(os.path.abspath("../"))

from pathlib import Path

from util import dataset as u_dataset
from util import visualization as u_visualization

In [ ]:
# Custom blue colormap
cmap = LinearSegmentedColormap.from_list("bwt", ["#f0f4ff", "#4e8df5", "#0b3d91"], N=256)


def plot_cm(timestamp, distance, candidates, object_name):
    # ── Helpers ───────────────────────────────────────────────────────────────────
    def class_labels(name, n):
        if name in [u_dataset.CategoryNames.BALL.value, "balls_seen_guessed", "balls_seen"]:
            return ["Kein Ball", "Ball"]
        if name == u_dataset.CategoryNames.PENALTYMARK.value:
            return ["Kein Elfmeterpunkt", "Elfmeterpunkt"]
        if name == u_dataset.CategoryNames.INTERSECTIONS.value:
            return ["None", "L", "T", "X"]
        return [str(i) for i in range(n)]

    def title_label(name, n):
        if name == u_dataset.CategoryNames.BALL.value:
            return "Ball"
        if name == u_dataset.CategoryNames.PENALTYMARK.value:
            return "Elfmeterpunkt"
        if name == u_dataset.CategoryNames.INTERSECTIONS.value:
            return "Linienkreuzungen"
        return [str(i) for i in range(n)]

    json_path = Path(
        "../../logs/fit/final",
        timestamp,
        "thresholded_metrics",
        f"d_{distance}-K_{candidates}.json",
    ).as_posix()

    # ── Load data from JSON file ─────────────────────────────────────────────────
    with open(json_path) as f:
        data = json.load(f)

    """
    Plot the confusion matrix for a single object.

    Parameters
    ----------
    data        : dict  — the full metrics dict loaded from JSON
    object_name : str   — key to look up, e.g. "ball", "penaltyMark", "intersections"
    """
    if object_name not in data:
        raise KeyError(f"'{object_name}' not found in data. Available keys: {list(data.keys())}")

    entry = data[object_name]
    cm = entry["confusion_matrix"]
    precision = entry["precisions"]
    recall = entry["recalls"]
    cm_arr = np.array(cm)
    n = len(cm_arr)
    labels = class_labels(object_name, n)

    # Figure size scales with matrix size
    fig_size = max(5, n * 1.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size), facecolor="#f7f9fc")
    ax.set_facecolor("#f7f9fc")

    # Row-normalise
    row_sums = cm_arr.sum(axis=1, keepdims=True)
    cm_norm = np.where(row_sums > 0, cm_arr / row_sums, 0.0)

    im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1, aspect="equal")

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, fontsize=10, fontweight="medium")
    # ax.set_yticklabels(labels, fontsize=10, fontweight="medium")
    ax.set_yticklabels(labels, fontsize=10, fontweight="medium", rotation=90, va="center")
    ax.set_xlabel("Vorhergesagt", fontsize=11, labelpad=8)
    ax.set_ylabel("Ground Truth", fontsize=11, labelpad=8)

    # Cell annotations
    thresh = 0.55
    counter = 0
    for i in range(n):
        for j in range(n):
            val = cm_arr[i, j]
            counter += val
            frac = cm_norm[i, j]
            color = "white" if frac > thresh else "#1a1a2e"
            ax.text(
                j,
                i,
                f"{val:,}\n({frac:.1%})",
                ha="center",
                va="center",
                fontsize=9,
                color=color,
                fontweight="bold" if i == j else "normal",
            )
    print(counter)
    # Diagonal highlight borders
    for k in range(n):
        rect = plt.Rectangle(
            (k - 0.5, k - 0.5),
            1,
            1,
            linewidth=2.2,
            edgecolor="white",
            facecolor="none",
        )
        ax.add_patch(rect)

    # Colourbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_ticks([])
    # cbar.ax.tick_params(labelsize=10)
    # cbar.set_label("Row-normalised proportion", fontsize=9)

    # Title
    avg_p = np.mean(precision) if isinstance(precision, list) else precision
    avg_r = np.mean(recall) if isinstance(recall, list) else recall
    ax.set_title(
        f"{title_label(object_name, n)} · Distanz {distance} m \nPrecision {avg_p:.3f}  ·  Recall {avg_r:.3f}",
        fontsize=12,
        fontweight="bold",
        pad=12,
        color="#1a1a2e",
    )

    save_path = "../../plots/confusion_matrices"
    os.makedirs(save_path, exist_ok=True)
    # plt.savefig(f"{save_path}/{object_name}.pdf")
    fig.tight_layout()
    plt.show()


timestamp = "20260711-000000"
distance = "9"
candidates = "5-4-11"


# plot_cm(timestamp, distance, candidates, u_dataset.CategoryN'ames.BALL.value)
# plot_cm(timestamp, distance, candidates, u_dataset.CategoryNames.PENALTYMARK.value)
# plot_cm(timestamp, distance, candidates, u_dataset.CategoryNames.INTERSECTIONS.value)'

In [ ]:
distance = 9

with open(f"../../logs/fit/final/20260711-000000/comparisons/d_{distance}-K_5-4-11.json") as f:
    data = json.load(f)

objects = [
    # "balls_seen",
    "balls_seen_guessed",
    # "penaltyMark",
    "intersections",
]

for obj in objects:
    u_visualization.plot_cm_comparison(data, obj, distance)

In [ ]:
def extract_binary(confusion_matrix):
    """Extracts TP, FP, FN from a 2x2 confusion matrix [[TN, FP], [FN, TP]]."""
    tn, fp = confusion_matrix[0]
    fn, tp = confusion_matrix[1]
    return tp, fp, fn


def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def format_binary_row(distance_m, category_data, category_data2= None):
    bh_tp, bh_fp, bh_fn = extract_binary(category_data["bhuman_confusion_matrix"])
    sys_tp, sys_fp, sys_fn = extract_binary(category_data["model_confusion_matrix"])

    if category_data2 is not None:
        bh_tp2, bh_fp2, bh_fn2 = extract_binary(category_data2["bhuman_confusion_matrix"])

        bh_p2 = precision(bh_tp2, bh_fp2) * 100
        bh_r2 = recall(bh_tp2, bh_fn2) * 100

        string = f"& {bh_tp2} & {bh_fp2} & {bh_fn2} & {bh_p2:.1f} & {bh_r2:.1f} "
    else:
        string = ""

    bh_p = precision(bh_tp, bh_fp) * 100
    bh_r = recall(bh_tp, bh_fn) * 100
    sys_p = precision(sys_tp, sys_fp) * 100
    sys_r = recall(sys_tp, sys_fn) * 100

    return (
        f"{distance_m}\\,m {string}"
        f"& {bh_tp} & {bh_fp} & {bh_fn} & {bh_p:.1f} & {bh_r:.1f} "
        f"& {sys_tp} & {sys_fp} & {sys_fn} & {sys_p:.1f} & {sys_r:.1f} \\\\"
    ).replace(".", "{,}")


def format_rows(distance_m, data):
    print(f"% ===== {distance_m} m =====")

    # print("% Ball (balls_seen+guessed):")
    # print(format_binary_row(distance_m, data["balls_seen"], data["balls_seen_guessed"]))

    print("% Elfmeterpunkt:")
    print(format_binary_row(distance_m, data["penaltyMark"]))


format_rows(distance_m=distance, data=data)